# Lesson 20 - Neural Networks for Classification

We will use Neural Networks for classification of a consumption patter for one day (frequency at 15 minutes) in one house. We have information about the consumption of each device. 

Our goal is to classify the device, based on the energy consumption patter.  




# Data  

Column `Class` has the labels / data we are using to "predict".  We can see there is an <mark> imbalance</mark> in the data! There are many more samples in classes 4, 3, and 1 compared to the others.  

Class  
4    2406  
1    2231  
3    1474   
2     851   
6     728  
0     727  
5     509    

There appears to be no missing data.  

## Train - Test Split  

Since the classes are imbalanced, we especially want to make sure we stratify when doing our TTS.  

# Neural Network Preprocessing  

When preprocessing, make sure we normalize data with a standard scaler, for example. Mean = 0 , sdev = 1.  

Remember to apply `fit_transform` to **BOTH** test and training data. This way, we are staying within the scale of mean 0 and sdev 1 intrinsic to each data set.  

Look at "Lesson 18/19" for in-depth explanation of each hyperparameter involved in MLP. One exception are these:

- **RELU**: rectified linear unit functions $f(x) = max(0, x)$.
- **IDENTITY**: no-op activation, useful to implement linear bottleneck $f(x) = x$.
- **LOGISTIC**: sigmoid function $f(x) = 1 / (1 + exp(-x))$.

# Imports

In [2]:
# %cd ..

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pickle

from Library import data

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from skopt import BayesSearchCV

# Get Data

In [4]:
df = data.get_data("Datasets/Consumption.csv", index_col=0)
df

,Class,H00:00,H00:15,H00:30,H00:45,H01:00,H01:15,H01:30,H01:45,H02:00,...,H21:30,H21:45,H22:00,H22:15,H22:30,H22:45,H23:00,H23:15,H23:30,H23:45
0,0,-0.18611,-0.18611,-0.18611,-0.18611,-0.18611,-0.18611,-0.18611,-0.18611,-0.18611,...,5.99020,2.91270,-0.18611,-0.18611,-0.18611,-0.18611,-0.18611,-0.18611,-0.18611,-0.18611
1,0,-0.17700,-0.17700,-0.17700,-0.17700,-0.17700,-0.17700,-0.17700,-0.17700,-0.17700,...,0.19151,6.47670,1.64510,-0.17700,-0.17700,-0.17700,-0.17700,-0.17700,-0.17700,-0.17700
2,0,-0.21353,-0.21353,-0.21353,-0.21353,-0.21353,-0.21353,-0.21353,-0.21353,-0.21353,...,-0.21353,-0.21353,-0.21353,-0.21353,-0.21353,-0.21353,-0.21353,-0.21353,-0.21353,-0.21353
3,0,-0.17147,-0.17147,-0.17147,-0.17147,-0.17147,-0.17147,-0.17147,-0.17147,-0.17147,...,0.16088,0.18043,0.16088,7.35540,1.17750,-0.17147,-0.17147,-0.17147,-0.17147,-0.17147
4,0,-0.16927,-0.16927,-0.16927,-0.16927,-0.16927,-0.16927,-0.16927,-0.16927,-0.16927,...,0.19095,6.07440,0.17093,0.19095,0.19095,7.39520,0.85133,-0.16927,-0.16927,-0.16927
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8921,4,-0.61198,-0.61198,-0.61198,-0.61198,-0.61198,-0.61198,-0.61198,-0.61198,-0.61198,...,1.79450,1.62850,1.71150,1.71150,1.71150,1.62850,-0.61198,-0.61198,-0.61198,-0.61198
8922,4,-0.49837,-0.49837,-0.49837,-0.49837,-0.49837,-0.49837,-0.49837,-0.49837,-0.49837,...,2.21890,2.31600,2.31600,2.21890,2.21890,1.73370,-0.49837,-0.49837,-0.49837,-0.49837
8923,4,-0.39939,-0.39939,-0.39939,-0.39939,-0.39939,-0.39939,-0.39939,-0.39939,-0.39939,...,2.64690,2.64690,2.64690,2.43680,2.64690,2.54190,0.44097,-0.39939,-0.39939,-0.39939
8924,4,-0.59560,-0.59560,-0.59560,-0.59560,-0.59560,-0.59560,-0.59560,-0.59560,-0.59560,...,1.95930,1.78310,1.78310,1.87120,1.78310,1.78310,1.78310,1.87120,1.87120,1.78310


## EDA

In [5]:
print(df.shape)
df.head()
df["Class"].value_counts()

# Classes (ordered as strings)
cls = [str(x) for x in sorted(df["Class"].unique())]
print(cls)

# Missing Data
for c in df.columns:
    if df[c].isna().sum() > 0:
        print(f"How many are missing? {c}, {df[c].isna().sum()}")

(8926, 97)
['0', '1', '2', '3', '4', '5', '6']


# Partition Dataset / Train-Test-Split Data

In [ ]:
from sklearn.model_selection import train_test_split

# Inputs and Outputs.
y = df["Class"]
X = df.loc[:, df.columns != "Class"]

# Train-Test-Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42, stratify=y)

# Neural Network Preprocessing

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_norm = scaler.fit_transform(X_train)
X_test_norm = scaler.fit_transform(X_test)

# Explore Transform
pd.DataFrame(X_train_norm, columns=X_train.columns).describe().T
pd.DataFrame(X_test_norm, columns=X_train.columns).describe().T



,count,mean,std,min,25%,50%,75%,max
H00:00,2946.0,-1.447134e-17,1.00017,-10.071015,-0.383661,-0.099596,0.031319,12.908714
H00:15,2946.0,2.291295e-17,1.00017,-5.822688,-0.389371,-0.088807,0.042614,13.509108
H00:30,2946.0,-1.205945e-17,1.00017,-3.121379,-0.435749,-0.097126,0.048195,6.107519
H00:45,2946.0,-1.567728e-17,1.00017,-2.925248,-0.415681,-0.081021,0.063848,10.880034
H01:00,2946.0,7.235669e-18,1.00017,-2.842828,-0.425455,-0.093400,0.049858,10.597514
...,...,...,...,...,...,...,...,...
H22:45,2946.0,-1.447134e-17,1.00017,-3.287286,-0.437196,-0.296507,0.230627,7.040024
H23:00,2946.0,2.110404e-17,1.00017,-4.689948,-0.410954,-0.253903,0.108274,9.315933
H23:15,2946.0,7.235669e-18,1.00017,-5.886643,-0.394194,-0.185832,-0.026952,8.306974
H23:30,2946.0,2.291295e-17,1.00017,-6.097248,-0.409583,-0.152961,-0.015241,7.821173


# Multilayer Perceptron Neural Network - Training

In [ ]:
from sklearn.neural_network import MLPClassifier
from skopt import BayesSearchCV
import pickle

params = {
    "hidden_layer_sizes": [10, 50, 100, 200, 300],
    "activation": ["relu", "identity", "logistic"],
    "alpha": [0.0001, 0.001, 0.01],
    "momentum": [0.95, 0.90, 0.85, 0.80],
    "learning_rate_init": [0.001, 0.01, 0.1], # initial learning rate per later
    "n_iter_no_change": [10, 20, 30, 40, 50], # no. iters with no improvement to wait before accepting convergence.
    "learning_rate": ["constant", "adaptive", "invscaling"], # rate types on how to change from initial learning rate.
}
# Early stopping reduces the adjustment time and decreases the possibilitey of over adjustment (over fitting). 
mlp = MLPClassifier(max_iter=10000, early_stopping=True, random_state=0)
mlp_bs = BayesSearchCV(mlp, params, n_iter=15, cv=5, n_jobs=-1, refit=True, random_state=0)
mlp_bs.fit(X_train_norm, y_train)

# Save
with open("Lesson 20 - Neural Networks for Classification/multilayer_perceptron_neural_network_bayes_search_cv.pkl", "wb") as file:
    pickle.dump(mlp_bs, file)

<>:20: SyntaxWarning: invalid escape sequence '\ '
<>:20: SyntaxWarning: invalid escape sequence '\ '
/var/folders/v3/j0_twvc12kb14hzn7ps4qy340000gn/T/ipykernel_27661/3534602240.py:20: SyntaxWarning: invalid escape sequence '\ '
  with open("Lesson\ 20\ -\ Neural\ Networks\ for\ Classification/multilayer_perceptron_neural_network_bayes_search_cv.pkl", "wb") as file:
/var/folders/v3/j0_twvc12kb14hzn7ps4qy340000gn/T/ipykernel_27661/3534602240.py:20: SyntaxWarning: invalid escape sequence '\ '
  with open("Lesson\ 20\ -\ Neural\ Networks\ for\ Classification/multilayer_perceptron_neural_network_bayes_search_cv.pkl", "wb") as file:


FileNotFoundError: [Errno 2] No such file or directory: 'Lesson\\ 20\\ -\\ Neural\\ Networks\\ for\\ Classification/multilayer_perceptron_neural_network_bayes_search_cv.pkl'